# 02. Loops

**Tier:** Intermediate. **Why you'd use this:** A signup form rejects a password that has no digit. You have to look at each character, stop when you find one, and say no only if the whole password was letters.

**After this lesson you can:** walk every character of a password; repeat while a condition is still true; tell a search that finished with no hit apart from a search that stopped early.
About 25 minutes. **Uses:** [Decisions](01-decisions-lesson.ipynb), [Indexing](../02-strings/01-indexing-lesson.ipynb)


## Quick review

1. Why is `"hello"[1:4]` three characters?
2. What does a `match` return when no arm fits and there is no `case _`?

<details><summary>Answers</summary>

The slice leaves out its stop. With no fitting arm and no `_`, the function returns `None`.
</details>


## The idea

`for` asks an object for its values, one at a time, and runs the body once per value. A string hands over characters. `range(n)` hands over `0` through `n - 1` and stops before `n`. `while` repeats as long as its condition is true, so the body has to change something that condition uses.

`else` on a `for` or `while` is not the `else` of an `if`. It runs when the loop ended because it ran out of values, or because the `while` condition became false. A `break` skips that `else`.

<details><summary>Explain more</summary>

`range(1, 6, 2)` starts at 1, stops before 6, and steps by 2: `1`, `3`, `5`. The stop is excluded, the same rule as a slice. An `int` is not a sequence of numbers. `for` cannot walk `3`.
</details>


## See it work

Accept the password only if some character is a digit.


In [1]:
def has_digit(password):
    # 1. One character at a time.
    for char in password:
        # 2. A digit is enough. Leave the loop.
        if "0" <= char <= "9":
            break
    else:
        # 3. else runs only when break never ran.
        return False
    return True


print(has_digit("ab3"))
print(has_digit("abc"))
print(has_digit(""))

True
False
False


## Check yourself

Answer these before you run the cell.

1. Does `range(5)` print `5`?
2. What does the `while` print, and why does `4` never get added?
3. What does `has_digit("")` return, and which clause ran?


In [2]:
for n in range(5):
    print(n)
for n in range(1, 6, 2):
    print(n)
total = 0
n = 1
while n <= 3:
    total = total + n
    n = n + 1
print(total)

0
1
2
3
4
1
3
5
6


<details><summary>Answers</summary>

`range(5)` prints `0` through `4`. The stop is left out. The `while` adds `1 + 2 + 3`, which is `6`. When `n` becomes `4` the condition is false. An empty password never breaks, so the `else` returns `False`.
</details>


## Why it works this way

`for` uses the iteration protocol. It asks the object for an iterator, then pulls the next value until the object says it is done. A string and a `range` both know how to do that. An `int` does not, so `for` over a number raises `TypeError`. Each `for` asks for a new iterator, so a second loop over the same password sees every character again. The walk does not use the string up. One saved iterator, pulled twice, would be empty the second time. Deep dive: [The iteration protocol](../../internals/07-iteration/01-protocol-lesson.ipynb).

`range` does not build the numbers. It stores a start, a stop, and a step, and computes a value when you ask. `range(10)` and `range(10**12)` are the same size. Pulling every value would still take time proportional to how many you pull. The object itself stays small either way. That is the point of excluding the stop: the range can describe "up to but not including n" without storing n numbers.

`else` exists so "finished with no `break`" has a place to go. It is not "the `if` was false." A `for` that never breaks always runs its `else`, even when the body found what you care about. A `while` runs its `else` when the condition becomes false, and skips it on `break`. The cost of `else` is nothing by itself. The cost is the loop: one step per value, and a `break` that skips the rest.

A `while` that reads until empty often uses `:=` so the condition assigns and tests in one place: `while (line := next_line()) != "":`.


In [3]:
import sys

print(sys.getsizeof(range(10)) == sys.getsizeof(range(10**12)))
print(10 in range(10))
word = "ab"
for char in word:
    print("first", char)
for char in word:
    print("second", char)

True
False
first a
first b
second a
second b


In [4]:
def bounded(n):
    while n < 3:
        if n == 1:
            break
        n = n + 1
    else:
        return "finished"
    return "broke"


print(bounded(0), bounded(5))

broke finished


## Reading until empty

`:=` in a `while` condition assigns the next chunk and decides whether to stop, so the body does not call the producer twice.


In [5]:
def chunks():
    # Stand-in for reading lines from a file or socket.
    return iter(["one", "two", ""])


stream = chunks()
parts = []
while (chunk := next(stream)) != "":
    parts.append(chunk)
print(parts)

['one', 'two']


## When to use it

Use `for` to walk a string or a `range`. Use `while` when you do not know how many steps you need, and the body can make the condition false. Don't use `while` to walk a string one character at a time. `for` already does that, and it cannot run past the end. Use `range` when you want the numbers. Use `for`/`else` when "never broke" is a real outcome, such as a search with no hit. After a loop that always runs to the end, plain code after the loop is clearer than `else`. Use `:=` in a `while` when one call both produces the next value and decides whether to stop.


## Common mistake

An `else` on `for` that is meant as "if this character was not a vowel." The `else` belongs to the loop. This loop never breaks, so the `else` always runs and replaces the vowels with `"none"`.


In [6]:
def vowels(word):
    found = ""
    for char in word:
        if char in "aeiou":
            found = found + char
    else:
        found = "none"
    return found


print(vowels("sky a"))

none


`else` is the no-`break` path, not the failed `if`. This search never needs `break`, so the check belongs after the loop. An empty `found` means there was no vowel. A word that has one keeps it.


In [7]:
def vowels(word):
    found = ""
    for char in word:
        if char in "aeiou":
            found = found + char
    if found == "":
        return "none"
    return found


print(vowels("sky a"))
print(vowels("sky"))

a
none


Looping over the number itself is a different bug, and it raises. Read the last line first: `TypeError`. The arrow marks the `for`. An `int` has no sequence of values. `range` is what you walk when you want to count.


In [8]:
def walk(n):
    for step in n:
        return step


print(walk(3))

TypeError: 'int' object is not iterable

## Try it

Change `has_digit` so it returns the first digit it finds, or `""` when there is none. The `""` has to live in the `else`. Why would that result be wrong if you assigned `""` on the line after the loop instead?


## Recap

- `for` pulls values from a string or a `range`. `range(n)` stops before `n`.
- `while` repeats until its condition is false. The body must change that condition.
- `else` on a loop runs only when the loop did not `break`.

- `while (item := …)` assigns and tests in one condition.
**Next:** [practice](02-loops-practice.ipynb)
